In [1]:
import re
import pandas as pd
from pathlib import Path

# ============================================================
# Config (set BOTH dirs)
# ============================================================
LABELED_DIR = Path("/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/ioki/ml-service_now_tagging/data/tagged/")       # 01-Dec-2025-result.xlsx ... 31-Dec-2025-result.xlsx
UNLABELED_DIR = Path("/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/ioki/ml-service_now_tagging/data/untagged/")   # 01-Dec.xlsx ... 31-Dec.xlsx

LABELED_PATTERN = "*-Dec-2025-result.xlsx"
UNLABELED_PATTERN = "*-Dec.xlsx"

OUTPUT_CSV = "merged_labeled_dec_2025_with_titles_and_levels.csv"


# ============================================================
# Helpers
# ============================================================
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

def load_and_merge_excels(folder: Path, pattern: str, add_source=True) -> pd.DataFrame:
    dfs = []
    for file_path in sorted(folder.glob(pattern)):
        try:
            part = pd.read_excel(file_path)
            part = standardize_columns(part)
            if add_source:
                part["source_file"] = file_path.name
            dfs.append(part)
        except Exception as e:
            print(f"⚠️ Skipping {file_path.name}: {e}")

    if not dfs:
        raise FileNotFoundError(f"No files matched pattern '{pattern}' in {folder}")

    return pd.concat(dfs, ignore_index=True)

def normalize_label(x) -> str:
    """Drop leading 1a/2g/3 prefix, lowercase, remove punctuation, collapse whitespace."""
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = re.sub(r"^\s*\d+[a-z]?\s*", "", s, flags=re.IGNORECASE)  # remove '1a'/'2g'/'3'
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)  # punctuation -> space
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_predicted_tags(predicted_tags: str):
    """
    Expected format:
      "(<L1>, <L2>), <L3>"
    Example:
      "(1g Self service, 2g Webmail), 3 technical issues"
    """
    if pd.isna(predicted_tags):
        return ("", "", "")

    s = str(predicted_tags).strip()

    m = re.search(r"^\s*\((.*?)\)\s*,\s*(.+?)\s*$", s)
    if not m:
        return ("", "", "")

    inside = m.group(1)   # "<L1>, <L2>"
    l3_raw = m.group(2)   # "<L3>"

    parts = [p.strip() for p in inside.split(",", 1)]
    l1_raw = parts[0] if len(parts) > 0 else ""
    l2_raw = parts[1] if len(parts) > 1 else ""

    return (normalize_label(l1_raw), normalize_label(l2_raw), normalize_label(l3_raw))


# ============================================================
# 1) Load labeled files (many rows; may miss title)
# ============================================================
df_labeled = load_and_merge_excels(LABELED_DIR, LABELED_PATTERN, add_source=True)

# Ignore Tag 1 if present
df_labeled = df_labeled.drop(columns=["tag_1"], errors="ignore")

# Validate required columns in labeled
required_labeled = ["number", "description", "predicted_tags"]
missing = [c for c in required_labeled if c not in df_labeled.columns]
if missing:
    raise KeyError(f"Missing required labeled columns: {missing}. Available: {list(df_labeled.columns)}")

# Keep only relevant labeled columns (title may or may not exist)
keep_labeled = ["number", "title", "description", "predicted_tags", "source_file"]
keep_labeled = [c for c in keep_labeled if c in df_labeled.columns]
df_labeled = df_labeled[keep_labeled]

print("Labeled rows:", len(df_labeled))
print("Labeled cols:", list(df_labeled.columns))


# ============================================================
# 2) Load unlabeled files (for titles)
# ============================================================
df_unlabeled = load_and_merge_excels(UNLABELED_DIR, UNLABELED_PATTERN, add_source=False)

# Validate required columns in unlabeled
required_unlabeled = ["number", "title"]
missing = [c for c in required_unlabeled if c not in df_unlabeled.columns]
if missing:
    raise KeyError(f"Missing required unlabeled columns: {missing}. Available: {list(df_unlabeled.columns)}")

# Build a unique (number -> title) mapping (keep first occurrence)
df_titles = (
    df_unlabeled[["number", "title"]]
    .dropna(subset=["number"])
    .drop_duplicates(subset=["number"], keep="first")
)

print("Unique ticket titles available:", len(df_titles))


# ============================================================
# 3) Merge titles into labeled (left join)
#    - keep labeled row count
#    - fill missing/empty labeled titles with unlabeled title
# ============================================================
df_merged = df_labeled.merge(
    df_titles,
    on="number",
    how="left",
    suffixes=("", "_from_unlabeled"),
)

# If labeled had a title column: fill missing/empty
if "title" in df_merged.columns:
    df_merged["title"] = df_merged["title"].replace(r"^\s*$", pd.NA, regex=True)
    if "title_from_unlabeled" in df_merged.columns:
        df_merged["title"] = df_merged["title"].fillna(df_merged["title_from_unlabeled"])
        df_merged = df_merged.drop(columns=["title_from_unlabeled"])
else:
    # labeled had no title col at all, so rename the joined column
    df_merged = df_merged.rename(columns={"title_from_unlabeled": "title"})

print(f"Missing title rate after join: {df_merged['title'].isna().mean():.2%}")


# ============================================================
# 4) Parse predicted_tags into level1/2/3 true labels
# ============================================================
df_merged["level1_true"], df_merged["level2_true"], df_merged["level3_true"] = zip(
    *df_merged["predicted_tags"].apply(parse_predicted_tags)
)

print("Parsed label coverage (% non-empty):")
print(
    pd.Series({
        "level1_true": (df_merged["level1_true"] != "").mean(),
        "level2_true": (df_merged["level2_true"] != "").mean(),
        "level3_true": (df_merged["level3_true"] != "").mean(),
    }).mul(100).round(2)
)


# ============================================================
# 5) Save final CSV
# ============================================================
df_merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Saved merged CSV to: {OUTPUT_CSV}")

df_merged.head()


Labeled rows: 23659
Labeled cols: ['number', 'title', 'description', 'predicted_tags', 'source_file']
Unique ticket titles available: 7886
Missing title rate after join: 0.00%
Parsed label coverage (% non-empty):
level1_true    100.00
level2_true     89.18
level3_true    100.00
dtype: float64
Saved merged CSV to: merged_labeled_dec_2025_with_titles_and_levels.csv


,number,title,description,predicted_tags,source_file,level1_true,level2_true,level3_true
0,INC0133168,Support - YouSee mail/Issue with Mit YouSee/Ca...,When the Customer tries to log in to his email...,"(1g Self service, 2g Webmail), 3 Login issues",01-Dec-2025-result.xlsx,self service,webmail,login issues
1,INC0133168,Support - YouSee mail/Issue with Mit YouSee/Ca...,When the Customer tries to log in to his email...,"(1g Self service, 2g Webmail), 3 Login issues",01-Dec-2025-result.xlsx,self service,webmail,login issues
2,INC0133168,Support - YouSee mail/Issue with Mit YouSee/Ca...,When the Customer tries to log in to his email...,"(1g Self service, 2g Webmail), 3 Login issues",01-Dec-2025-result.xlsx,self service,webmail,login issues
3,INC0133171,Support - TV/Video on demand/Streamer or Audio,Customer experiences that when they get up at ...,"(1c TV, 2c OTT), 3 technical issues",01-Dec-2025-result.xlsx,tv,ott,technical issues
4,INC0133171,Support - TV/Video on demand/Streamer or Audio,Customer experiences that when they get up at ...,"(1c TV, 2c OTT), 3 technical issues",01-Dec-2025-result.xlsx,tv,ott,technical issues
